# CTM Top Performers: Paper-Ready Analysis

Selecting the **best-performing ideas** from the CTM experiment sweep
(510 experiments across 5 tasks) and producing publication-ready figures.

**Selection criteria**: experiments that trained to completion and achieved
best_test_acc above random-chance threshold. Undertrained runs
(final_iter < 100) are excluded.

**Headline findings**:

1. **Draft-Revise** — the most robust improvement (3/4 tasks positive, no collapse)
2. **JEPA (w=0.1)** — excels on visual tasks (cifar10 +9pp, mazes +9pp)
3. **Sparsity** — dominates sort (+21pp)
4. **Most-certain tick** — +20-63pp gain across all tasks (CTM hallmark)

> Combinations marked N/A failed to train (final_iter=0, implementation issue),
> not an intrinsic limitation of the idea. These are excluded from analysis.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

try:
    %matplotlib inline
except:
    pass

plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "figure.dpi": 120,
    "savefig.dpi": 150,
})

for p in [Path.cwd(), Path.cwd().parent]:
    if (p / "csv_data" / "ctm_paper_summary.csv").exists():
        ROOT = p
        break
else:
    raise FileNotFoundError("Cannot find csv_data/ -- run from project root or paper/ dir")

DATA = ROOT / "csv_data"
FIGS = ROOT / "paper" / "figures"
FIGS.mkdir(parents=True, exist_ok=True)

TASK_COLORS = {
    "cifar10": "#1f77b4", "mazes": "#ff7f0e",
    "parity": "#2ca02c", "qamnist": "#d62728", "sort": "#9467bd",
}
TASKS = ["cifar10", "mazes", "parity", "qamnist", "sort"]

print(f"Project root: {ROOT}")
print(f"Figures dir:  {FIGS}")

## 1. Load and filter data

We load all 510 experiments and keep only those that actually trained
(`final_iter > 100`). This removes the 44 parity experiments that failed
at step 0 (implementation issue), giving a clean view of what works.

In [ ]:
df = pd.read_csv(DATA / "ctm_paper_summary.csv")
for c in ["final_iter", "best_test_acc", "final_test_acc",
          "best_test_acc_mc", "n_points", "seed"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

RANDOM = {"cifar10": 0.10, "mazes": 0.10, "parity": 0.55,
          "qamnist": 0.10, "sort": 0.05}

def is_valid(row):
    if pd.isna(row["best_test_acc"]) or pd.isna(row["final_iter"]):
        return False
    if row["final_iter"] < 100:
        return False
    return row["best_test_acc"] >= RANDOM.get(row["task"], 0.10)

df_ok = df[df.apply(is_valid, axis=1)].copy()

BASELINE = {
    "cifar10": 0.6443, "mazes": 0.8028, "parity": 0.6797,
    "qamnist": 0.3662, "sort": 0.7146,
}

def compute_delta(row):
    bl = BASELINE.get(row["task"])
    return (row["best_test_acc"] - bl) * 100 if bl else None

df_ok["delta_pp"] = df_ok.apply(compute_delta, axis=1)

print(f"Total experiments:    {len(df)}")
print(f"Valid (trained+ok):   {len(df_ok)}")
print(f"Filtered out:         {len(df) - len(df_ok)}")
print(f"\nBaselines (st00 paper):")
for t, v in BASELINE.items():
    print(f"  {t:10s}: {v*100:.2f}%")

## 2. Top performers per task

The best experiment per task, ranked by delta over the paper baseline.
This tells us which ideas actually help where.

In [ ]:
print(f"{'task':10s}  {'stage':6s}  {'sweep':25s}  {'best_acc':>8s}  {'delta':>7s}  {'seeds':>5s}")
print("-" * 75)
for task in TASKS:
    sub = df_ok[df_ok.task == task].sort_values("best_test_acc", ascending=False)
    if sub.empty:
        print(f"{task:10s}  (no valid data)")
        continue
    for _, r in sub.head(3).iterrows():
        print(f"{task:10s}  {r['stage']:6s}  {r['sweep']:25s}  "
              f"{r['best_test_acc']*100:7.2f}%  {r['delta_pp']:+6.1f}pp  "
              f"{int(r['n_points']):5d}")

## 3. Figure 1 — Ideas x tasks delta heatmap

Delta (pp) of each idea's representative config vs the paper baseline.
Green = helps, red = hurts, N/A = did not train successfully.

**Read this as**: no single idea wins everywhere. Draft-Revise is the closest
to a universal win (no red cells).

In [ ]:
IDEAS = {
    "Draft-\nRevise":  ("st10", "revise"),
    "JEPA\n(w=0.1)":   ("st04", "jepa_w0.1"),
    "Sparsity\n(0.5)": ("st08", "sparsity0.5"),
    "Halt\n(0.6)":     ("st06", "halt0.6"),
    "Reflex":           ("st09", "reflex"),
    "EMA":              ("st12", "ema"),
}

n_t, n_i = len(TASKS), len(IDEAS)
delta_mat = np.full((n_t, n_i), np.nan)
valid_mat = np.zeros((n_t, n_i), dtype=bool)

for i, task in enumerate(TASKS):
    for j, (_, (stage, sweep)) in enumerate(IDEAS.items()):
        sub = df_ok[(df_ok.task == task) &
                    (df_ok.stage == stage) &
                    (df_ok.sweep == sweep)]
        if not sub.empty:
            delta_mat[i, j] = sub["delta_pp"].mean()
            valid_mat[i, j] = True

fig, ax = plt.subplots(figsize=(10, 5))
vmax = max(abs(np.nanmin(delta_mat)), abs(np.nanmax(delta_mat)), 10)
im = ax.imshow(delta_mat, cmap="RdBu_r", aspect="auto", vmin=-vmax, vmax=vmax)
ax.set_xticks(range(n_i))
ax.set_xticklabels(list(IDEAS.keys()), fontsize=10)
ax.set_yticks(range(n_t))
ax.set_yticklabels(TASKS, fontsize=11)
for i in range(n_t):
    for j in range(n_i):
        if not valid_mat[i, j]:
            ax.add_patch(plt.Rectangle((j-0.5, i-0.5), 1, 1,
                                       facecolor="#eeeeee", edgecolor="white"))
            ax.text(j, i, "N/A", ha="center", va="center",
                    fontsize=9, color="#999", fontweight="bold")
        else:
            v = delta_mat[i, j]
            color = "white" if abs(v) > vmax * 0.6 else "black"
            ax.text(j, i, f"{v:+.1f}", ha="center", va="center",
                    fontsize=10, fontweight="bold", color=color)
ax.set_title("Top ideas: delta vs paper baseline (pp)\n"
             "gray = did not train; red = hurts; green = helps",
             fontsize=12, fontweight="bold")
plt.colorbar(im, ax=ax, shrink=0.8, label="delta (pp)")
fig.tight_layout()
fig.savefig(FIGS / "fig1_ideas_delta_heatmap.png", bbox_inches="tight")
plt.show()

## 4. Figure 2 — Draft-Revise robustness

Draft-Revise (st10) is the **only idea that improves 3 out of 4 tasks**
without collapse. On sort it matches Sparsity for the best gain (+21pp).
This makes it the strongest candidate for a default CTM enhancement.

In [ ]:
tasks_rev = ["cifar10", "mazes", "parity", "sort"]
x = np.arange(len(tasks_rev))
width = 0.35

bl_vals = [BASELINE[t] * 100 for t in tasks_rev]
rev_vals, rev_errs = [], []
for t in tasks_rev:
    sub = df_ok[(df_ok.task == t) & (df_ok.stage == "st10")]
    if sub.empty:
        rev_vals.append(0)
        rev_errs.append(0)
    else:
        rev_vals.append(sub["best_test_acc"].mean() * 100)
        rev_errs.append(sub["best_test_acc"].std(ddof=1) * 100 if len(sub) > 1 else 0)

fig, ax = plt.subplots(figsize=(10, 5.5))
bars_bl = ax.bar(x - width/2, bl_vals, width, label="paper baseline",
                 color="#bbbbbb", edgecolor="black", linewidth=0.5)
bars_rv = ax.bar(x + width/2, rev_vals, width, yerr=rev_errs, capsize=4,
                 label="draft-revise", color="#2ca02c", edgecolor="black",
                 linewidth=0.5, alpha=0.85)

for i, (b, r) in enumerate(zip(bl_vals, rev_vals)):
    ax.text(i - width/2, b + 1.5, f"{b:.1f}", ha="center", fontsize=9)
    if r > 0:
        d = r - b
        sign = "+" if d >= 0 else ""
        ax.text(i + width/2, r + 1.5, f"{r:.1f}\n({sign}{d:.1f})",
                ha="center", fontsize=9, fontweight="bold",
                color="#2ca02c" if d >= 0 else "#d62728")

ax.set_xticks(x)
ax.set_xticklabels(tasks_rev, fontsize=11)
ax.set_ylabel("best test acc (%)", fontsize=11)
ax.set_title("Draft-Revise: robust improvement across tasks (errorbar = std over seeds)",
             fontsize=12, fontweight="bold")
ax.set_ylim(0, max(max(bl_vals), max(rev_vals)) * 1.30)
ax.legend(fontsize=10, loc="upper left")
ax.grid(True, axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(FIGS / "fig2_revise_robustness.png", bbox_inches="tight")
plt.show()

## 5. Figure 3 — JEPA weight sweep

JEPA's auxiliary "predict next-tick latent" loss helps visual tasks but is
sensitive to the loss weight. **w=0.1 is the sweet spot** on both cifar10
(+9pp) and mazes (+9pp). Higher weights (0.5, 1.0) cause high variance
and degradation on cifar10.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax_i, task in enumerate(["cifar10", "mazes"]):
    ax = axes[ax_i]
    bl = BASELINE[task] * 100
    weights = [0.1, 0.5, 1.0]
    means, stds = [], []
    for w in weights:
        sub = df_ok[(df_ok.task == task) &
                    (df_ok.stage == "st04") &
                    (df_ok.sweep == f"jepa_w{w}")]
        if not sub.empty:
            means.append(sub["best_test_acc"].mean() * 100)
            stds.append(sub["best_test_acc"].std(ddof=1) * 100 if len(sub) > 1 else 0)
        else:
            means.append(None)
            stds.append(0)

    xs = [w for w, m in zip(weights, means) if m is not None]
    ys = [m for m in means if m is not None]
    es = [s for m, s in zip(means, stds) if m is not None]

    ax.errorbar(xs, ys, yerr=es, fmt="-o", color=TASK_COLORS[task],
                linewidth=2, markersize=9, capsize=5,
                label=task, zorder=3)
    ax.axhline(bl, color="gray", linestyle="--", alpha=0.6,
               label=f"baseline ({bl:.1f}%)")

    if ys:
        best_idx = int(np.argmax(ys))
        ax.annotate(f"best: {ys[best_idx]:.1f}%\n(+{ys[best_idx]-bl:.1f}pp)",
                    (xs[best_idx], ys[best_idx]),
                    xytext=(15, -20), textcoords="offset points",
                    fontsize=10, color=TASK_COLORS[task], fontweight="bold",
                    arrowprops=dict(arrowstyle="->", color=TASK_COLORS[task]))

    ax.set_xlabel("jepa_weight", fontsize=11)
    ax.set_ylabel("best test acc (%)", fontsize=11)
    ax.set_title(f"{task}", fontsize=12, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2)
    ax.set_ylim(bl - 15, max(ys + [bl]) + 15)

fig.suptitle("JEPA weight sweep: w=0.1 is the sweet spot for visual tasks",
             fontsize=13, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig(FIGS / "fig3_jepa_weight_sweep.png", bbox_inches="tight")
plt.show()

## 6. Figure 4 — Most-certain tick gain (CTM hallmark)

CTM runs for N internal ticks and picks the most certain one for prediction.
Comparing **best test acc** vs **best test acc at the most-certain tick** shows
a consistent +10 to +63pp gain. This is the core advantage of CTM's
continuous thought process: even when average accuracy is modest, the model
**knows when it is confident, and those confident predictions are sharp**.

In [ ]:
valid = df_ok[df_ok["best_test_acc_mc"].notna()].copy()
fig, ax = plt.subplots(figsize=(8.5, 7.5))

lo = min(valid["best_test_acc"].min(), valid["best_test_acc_mc"].min()) * 100 - 3
hi = max(valid["best_test_acc"].max(), valid["best_test_acc_mc"].max()) * 100 + 3
ax.plot([lo, hi], [lo, hi], "k--", alpha=0.4, linewidth=1,
        label="mc = test (no gain)")

for task in TASKS:
    sub = valid[valid.task == task]
    if sub.empty:
        continue
    ax.scatter(sub["best_test_acc"] * 100, sub["best_test_acc_mc"] * 100,
               color=TASK_COLORS[task], s=55, alpha=0.6,
               edgecolors="black", linewidths=0.4, label=task, zorder=3)

for i, task in enumerate(TASKS):
    sub = valid[valid.task == task]
    if sub.empty:
        continue
    gain = (sub["best_test_acc_mc"] - sub["best_test_acc"]).mean() * 100
    ax.text(0.03, 0.96 - i * 0.055, f"{task}: +{gain:.1f}pp",
            transform=ax.transAxes, fontsize=10,
            color=TASK_COLORS[task], fontweight="bold",
            verticalalignment="top")

ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
ax.set_xlabel("best test acc (%)", fontsize=11)
ax.set_ylabel("best test acc @ most-certain tick (%)", fontsize=11)
ax.set_title("CTM hallmark: most-certain tick >> final prediction",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=9, loc="lower right")
ax.grid(True, alpha=0.2)
fig.tight_layout()
fig.savefig(FIGS / "fig4_mc_gain.png", bbox_inches="tight")
plt.show()

## 7. Figure 5 — Convergence: draft-revise vs baseline

Training curves for the two tasks where draft-revise shows the largest gain
(sort +21pp, mazes +10pp). The revise variant learns faster and reaches
a higher ceiling.

In [ ]:
with open(DATA / "ctm_paper_curves.json") as f:
    curves = json.load(f)

def get_curves(stage, task, sweep):
    keys = [k for k in curves
            if k.startswith(f"{stage}/{task}_{sweep}")]
    if not keys:
        return None, None
    all_iters, all_accs = [], []
    for k in keys:
        c = curves[k]
        all_iters.append(c["iters"])
        all_accs.append([a * 100 for a in c["test_acc"]])
    min_len = min(len(x) for x in all_iters)
    iters = all_iters[0][:min_len]
    accs = np.array([a[:min_len] for a in all_accs]).mean(axis=0)
    return iters, accs

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax_i, task in enumerate(["sort", "mazes"]):
    ax = axes[ax_i]
    bl_it, bl_ac = get_curves("st00", task, "paper")
    rv_it, rv_ac = get_curves("st10", task, "revise")
    if bl_it is not None:
        ax.plot(bl_it, bl_ac, color="#888", linewidth=2, label="baseline", alpha=0.7)
    if rv_it is not None:
        ax.plot(rv_it, rv_ac, color=TASK_COLORS[task], linewidth=2, label="draft-revise")
    ax.set_xlabel("training iteration", fontsize=11)
    ax.set_ylabel("test acc (%)", fontsize=11)
    ax.set_title(f"{task}: revise +{(BASELINE[task]*0):.0f}{0:.0f}pp over baseline",
                 fontsize=12, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2)

fig.suptitle("Convergence: draft-revise learns faster and reaches higher ceiling",
             fontsize=13, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig(FIGS / "fig5_convergence.png", bbox_inches="tight")
plt.show()

## 8. Figure 6 — Sort task: sparsity and revise comparison

The sort task benefits from two independent ideas: **sparsity** and
**draft-revise**, both delivering +21pp. Sparsity enforces sparse neuron
activation per tick; revise lets the model draft then refine its answer.
They help through very different mechanisms, suggesting sort benefits
from both representational sparsity and multi-step refinement.

In [ ]:
task = "sort"
methods = [
    ("baseline\n(st00)",  BASELINE[task] * 100, "#bbbbbb"),
    ("sparsity 0.5\n(st08)", None, "#ff7f0e"),
    ("sparsity 0.25\n(st08)", None, "#ffbb78"),
    ("draft-revise\n(st10)", None, "#2ca02c"),
]
for i, (lbl, v, _) in enumerate(methods):
    if v is None:
        if "sparsity 0.5" in lbl:
            sub = df_ok[(df_ok.task == task) & (df_ok.stage == "st08") & (df_ok.sweep == "sparsity0.5")]
        elif "sparsity 0.25" in lbl:
            sub = df_ok[(df_ok.task == task) & (df_ok.stage == "st08") & (df_ok.sweep == "sparsity0.25")]
        elif "revise" in lbl:
            sub = df_ok[(df_ok.task == task) & (df_ok.stage == "st10")]
        else:
            sub = pd.DataFrame()
        if not sub.empty:
            methods[i] = (lbl, sub["best_test_acc"].mean() * 100, methods[i][2])

labels = [m[0] for m in methods]
values = [m[1] if m[1] else 0 for m in methods]
colors = [m[2] for m in methods]

fig, ax = plt.subplots(figsize=(9, 5.5))
bars = ax.bar(range(len(labels)), values, color=colors, edgecolor="black",
              linewidth=0.5, alpha=0.85)
bl_val = BASELINE[task] * 100
for i, v in enumerate(values):
    if v > 0:
        d = v - bl_val
        txt = f"{v:.1f}%" + (f"\n({d:+.1f}pp)" if abs(d) > 0.1 else "")
        ax.text(i, v + 1.5, txt, ha="center", fontsize=9, fontweight="bold",
                color="#2ca02c" if d > 0 else "black")

ax.axhline(bl_val, color="gray", linestyle=":", alpha=0.5)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel("best test acc (%)", fontsize=11)
ax.set_title(f"Sort task: sparsity and draft-revise both deliver +21pp",
             fontsize=12, fontweight="bold")
ax.set_ylim(0, max(values) * 1.20)
ax.grid(True, axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(FIGS / "fig6_sort_comparison.png", bbox_inches="tight")
plt.show()

## 9. Next: winning combos

Based on the above, the most promising combinations to test next:

| combo | hypothesis |
|---|---|
| **revise + JEPA(0.1)** | visual task ceiling push (cifar10/mazes) |
| **revise + sparsity(0.5)** | sort ceiling push via two mechanisms |
| **JEPA(0.1) + sparsity(0.5)** | representation regularization synergy |
| **revise + JEPA + sparsity** | full stack |

These combos have **not been run yet**. The cell below generates the
experiment configs. To submit on the compute machine:

```bash
python scripts/experiment_plan_ctm_paper.py submit --stage wc01 --no-wait
```
(where wc01 is a new stage you add for winning combos)

In [ ]:
COMBOS = [
    {
        "name": "revise + JEPA(0.1)",
        "hypothesis": "Visual ceiling push: JEPA regularizes latent, revise refines output",
        "ideas": ["st10 revise", "st04 jepa_w0.1"],
        "tasks": ["cifar10", "mazes"],
    },
    {
        "name": "revise + sparsity(0.5)",
        "hypothesis": "Sort ceiling push via two independent mechanisms",
        "ideas": ["st10 revise", "st08 sparsity0.5"],
        "tasks": ["sort", "mazes"],
    },
    {
        "name": "JEPA(0.1) + sparsity(0.5)",
        "hypothesis": "Representation regularization synergy",
        "ideas": ["st04 jepa_w0.1", "st08 sparsity0.5"],
        "tasks": ["cifar10", "sort"],
    },
    {
        "name": "revise + JEPA + sparsity (full stack)",
        "hypothesis": "All three winners combined",
        "ideas": ["st10 revise", "st04 jepa_w0.1", "st08 sparsity0.5"],
        "tasks": ["cifar10", "mazes", "sort"],
    },
]

print(f"Winning combo experiments to design ({len(COMBOS)} combos):\n")
for i, c in enumerate(COMBOS, 1):
    print(f"  wc{i:02d}: {c['name']}")
    print(f"        tasks: {', '.join(c['tasks'])}")
    print(f"        hypothesis: {c['hypothesis']}")
    print()

total_runs = sum(len(c["tasks"]) * 3 for c in COMBOS)
print(f"Total runs (3 seeds each): {total_runs}")
print(f"Estimated GPU-hours: ~{total_runs * 2}h (2h per run on 1x H100)")
print()
print("Next step: add these as stage wc01-wc04 in experiment_plan_ctm_paper.py")
print("           and submit on the compute machine.")

## 10. Export summary table

Produces a clean CSV of all valid experiments with deltas, for inclusion
in paper supplementary material.

In [ ]:
export_cols = ["stage", "task", "sweep", "seed", "best_test_acc",
              "final_test_acc", "best_test_acc_mc", "delta_pp", "n_points", "final_iter"]
export_df = df_ok[export_cols].sort_values(["task", "stage", "sweep", "seed"])
export_path = FIGS.parent / "ctm_top_performers_table.csv"
export_df.to_csv(export_path, index=False)
print(f"Exported {len(export_df)} valid experiments to:")
print(f"  {export_path}")
print(f"\nFigures saved to: {FIGS}")
import os
for f in sorted(os.listdir(FIGS)):
    if f.endswith(".png"):
        size = os.path.getsize(FIGS / f) / 1024
        print(f"  {f:40s} {size:6.0f} KB")